# Inventaris bulanan, WIP, dan pelepasan biaya sapi

Cakupan Januari 2024–Desember 2025 untuk 15.000 sapi pada data simulasi Highwood Feeders. Tanggal masuk termasuk stok akhir hari dan tanggal keluar mengeluarkan sapi dari stok pada hari yang sama. Biaya produksi menggunakan hari masuk dan hari keluar secara inklusif. Pen mengikuti pen tercatat pada lot.

WIP berbasis sumber mencakup harga perolehan dan tindakan veteriner bertanggal. Estimasi WIP menambahkan alokasi pakan terkirim dan akrual yardage. Nilai aktual lengkap dan harga pasar tetap kosong ketika sumber yang diperlukan belum tersedia. [Metode, audit sumber, dan batas penggunaan](../phase2b/04B_MONTHLY_INVENTORY_WIP_FINDINGS.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from feedlot_analytics.exports import write_build
from feedlot_analytics.inventory_market import QUOTE_COLUMNS
from feedlot_analytics.monthly_inventory import build_monthly_inventory

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "phase2a/reconciliation.json").is_file())
QUOTE_ROWS = []  # Isi daftar dict berisi harga liveweight berizin, dengan kolom QUOTE_COLUMNS.
MAX_WEIGHT_AGE_DAYS = 45
EXPORT_TO = None  # Misalnya ROOT / "work/phase2b45-inventory".

result = build_monthly_inventory(ROOT, quote_rows=QUOTE_ROWS,
                                 max_weight_age_days=MAX_WEIGHT_AGE_DAYS)
tables = result.tables
display(pd.Series(result.reconciliation["summary"]).drop("table_rows").to_frame("hasil"))
display(pd.Series(result.reconciliation["summary"]["table_rows"]).to_frame("baris"))

## Stok dan pergerakan

Stok awal + penempatan − penjualan − mortalitas = stok akhir. Jumlah stok akhir adalah saldo pada masing-masing tanggal laporan. Detail `animal_month` memuat ID sapi, breed, jenis kelamin, lot, pen tercatat, dan tanggal transaksi yang sudah terjadi.

In [ ]:
monthly = tables["monthly_yard"].set_index("month_end")
annual = tables["annual_yard"].set_index("year")
stock = ["opening_head", "placed_head", "sold_head", "mortality_head", "closing_head"]
display(monthly[stock])
display(annual[stock])

fig, (stock_axis, flow_axis) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout="constrained")
stock_axis.plot(monthly.index, monthly.closing_head, color="#244d65", marker=".", label="Stok akhir")
stock_axis.set_ylabel("Ekor dalam stok")
stock_axis.legend()
for column, label, color in (
    ("placed_head", "Penempatan", "#147552"),
    ("sold_head", "Penjualan", "#224d87"),
    ("mortality_head", "Mortalitas", "#b34b32"),
):
    flow_axis.plot(monthly.index, monthly[column], marker=".", color=color, label=label)
flow_axis.set_ylabel("Ekor per bulan")
flow_axis.legend(ncol=3)
plt.show()

## WIP dan pelepasan biaya

WIP awal + perolehan + biaya produksi + penyesuaian yang diterapkan − pelepasan penjualan − pelepasan mortalitas = WIP akhir. Nilai dasar veteriner tanpa tanggal dan selisih alokasi pakan terhadap closeout berada pada tabel audit retrospektif. Biaya dalam tabel di bawah ditampilkan dalam CAD.

In [ ]:
financial = [
    "estimated_opening_wip_cents", "purchase_cents", "direct_vet_cents",
    "estimated_feed_cents", "estimated_yardage_cents", "adjustments_applied_cents",
    "estimated_sold_cogs_cents", "estimated_mortality_loss_cents", "estimated_closing_wip_cents",
    "source_sold_release_cents", "source_mortality_release_cents", "source_closing_wip_cents",
    "actual_closing_wip_cents", "actual_sold_cogs_cents", "actual_mortality_loss_cents",
]
def in_cad(frame, columns):
    return (frame[columns] / 100).round(2).rename(
        columns=lambda name: name.removesuffix("_cents") + "_cad"
    )

display(in_cad(monthly, financial))
display(in_cad(annual, financial))

audit = tables["lifetime_closeout_audit"].assign(
    exit_year=lambda frame: frame.exit_date.dt.year,
)
comparison = [
    "estimated_sold_cogs_cents", "estimated_mortality_loss_cents", "booked_lifetime_total_cents",
    "undated_vet_base_cents", "booked_less_allocated_delivery_cents",
    "booked_less_modeled_yardage_cents",
]
display(in_cad(audit.groupby(["exit_year", "exit_type"])[comparison].sum(), comparison))

## Harga pasar dan pakan

Estimasi bobot memakai penimbangan terakhir yang sudah tersedia. Harga diisi pada `QUOTE_ROWS` menggunakan tanggal akhir bulan, tanggal observasi, tanggal publikasi, kelas dan batas bobot yang sesuai, satuan liveweight, sumber, dan hak penggunaan. Total per kelompok hanya tampil jika seluruh sapi dalam stok memperoleh estimasi. Biaya penjualan per ekor opsional menghasilkan indikasi nilai neto tersendiri.

Catatan pakan mendukung pelaporan pengiriman ke lot dan pen yang tercatat. Saldo gudang, nilai persediaan pakan, dan penyesuaiannya belum tersedia. [Skema ledger yang diusulkan](../phase2b/04B_MONTHLY_INVENTORY_WIP_FINDINGS.md#persediaan-pakan-gudang)

In [ ]:
market_counts = ["closing_head", "weight_eligible_head", "market_valued_head",
                 "market_less_selling_costs_valued_head"]
market_values = ["closing_purchase_basis_cents", "estimated_closing_wip_cents",
                 "estimated_market_value_cents", "indicative_market_less_costs_to_sell_cents"]
display(monthly[market_counts].join(in_cad(monthly, market_values)))
display(tables["animal_month"].weight_basis.value_counts().to_frame("animal_months"))
display(pd.DataFrame(tables["market_quote_assumptions"], columns=QUOTE_COLUMNS))

feed = tables["monthly_feed_delivery"].groupby("month_end").sum(numeric_only=True)
display(feed.drop(columns="delivery_cost_cents").join(in_cad(feed, ["delivery_cost_cents"])))

## Ekspor opsional

Isi `EXPORT_TO` lalu jalankan ulang untuk memperoleh tabel CSV, kontrak grain, hash sumber dan hasil kontrol dari sumber analitis yang sama. Gunakan ID sebagai teks, dan pertahankan nilai kosong saat mengimpor ke Excel, Power BI, Tableau atau HTML.

In [ ]:
if EXPORT_TO is not None:
    write_build(result, EXPORT_TO)
    print(EXPORT_TO)